In [1]:
# ============================================================================
# COSADAM NLP COMPREHENSIVE EXPERIMENTAL SUITE (BERT-base on AG News)
# Kaggle Offline Ready & Time-Optimized with FP16 and Multi-GPU Support
#
# ADDRESSING REVIEWERS' COMMENTS:
# ----------------------------------------------------------------------------
# [Reviewer 3]: "larger-scale challenging datasets, deeper architectures" 
#     -> Implemented BERT-base on AG News (120k samples).
# [Reviewer 3]: "multiple random seeds, statistical significance analysis" 
#     -> 3 random seeds, Paired T-Tests, FDR correction, and 95% CI implemented.
# [Reviewer 2]: "comparisons with recent SOTA (past 3 years)... e.g., sharpness-aware" 
#     -> Added Lion (2023) and SAM (Sharpness-Aware Minimization).
# [Reviewer 2]: "careful hyperparameter tuning to hit high accuracy" 
#     -> Added Quick HP Search ensuring equal tuning budget across all optimizers.
# [Reviewer 1]: "Claims regarding 'flat minima' lack empirical validation" 
#     -> Implemented `evaluate_loss_sharpness()` for Transformer architectures.
#
# EXPERT VISUALIZATION UPGRADE: Added Dual-Panel Learning Curves (Acc & Loss) 
# with ±1 Std-Dev shaded confidence bands for rigorous reporting.
# ============================================================================

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

import os, time, warnings, random, json, math, zipfile, gc
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.optimizer import Optimizer
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, Subset
from transformers import BertTokenizerFast, BertForSequenceClassification, get_cosine_schedule_with_warmup
from torch.cuda.amp import autocast, GradScaler
from sklearn.metrics import accuracy_score
from scipy.stats import ttest_rel, t as t_dist 
from statsmodels.stats.multitest import multipletests

# ============================================================================
# 🛡️ ENVIRONMENT & REPRODUCIBILITY
# ============================================================================
os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GLOBAL_START_TIME = time.time()

HP_SEARCH_SEED = 42
MAIN_SEEDS = [42, 123, 587]   

AG_NEWS_PATH = '/kaggle/input/datasets/amananandrai/ag-news-classification-dataset'
BERT_PATH = '/kaggle/input/datasets/virajjayant/bertbaseuncased/bert-base-uncased'

RESULTS_DIR = "/kaggle/working/results_cosadam_nlp"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, "plots"), exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, "latex_tables"), exist_ok=True)

HP_CHECKPOINT_FILE = os.path.join(RESULTS_DIR, "hp_checkpoint.json")
MAIN_CHECKPOINT_FILE = os.path.join(RESULTS_DIR, "main_checkpoint.json")

# ============================================================================
# ⚙️ CONFIGURATION
# ============================================================================
NLP_EPOCHS = 3                
BATCH_SIZE = 64               
MAX_LENGTH = 128              
GRADIENT_CLIP_VALUE = 1.0
WARMUP_RATIO = 0.1
KAGGLE_TIME_LIMIT = 11.2 * 3600 

HP_EPOCHS = 1                  
HP_SUBSET_RATIO = 0.15         
HP_LR_GRID = [2e-5, 3e-5]      
HP_WD_GRID = [0.01]            

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# ============================================================================
# 🧠 OPTIMIZERS
# ============================================================================
class CosAdam(optim.Optimizer):
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8,
                 weight_decay=0.01, alpha_cos=0.9, c=0.5):
        defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay, alpha_cos=alpha_cos, c=c)
        super(CosAdam, self).__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = closure() if closure is not None else None
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None: continue
                grad = p.grad
                state = self.state[p]

                if len(state) == 0:
                    state['step'] = 0
                    state['exp_avg'] = torch.zeros_like(p)
                    state['exp_avg_sq'] = torch.zeros_like(p)
                    state['prev_grad'] = torch.zeros_like(p)
                    state['s'] = 0.0  

                exp_avg, exp_avg_sq = state['exp_avg'], state['exp_avg_sq']
                prev_grad, s = state['prev_grad'], state['s']
                beta1, beta2 = group['betas']
                state['step'] += 1

                if state['step'] > 1:
                    cos_theta = F.cosine_similarity(grad.flatten(), prev_grad.flatten(), dim=0, eps=1e-8)
                    s = group['alpha_cos'] * s + (1 - group['alpha_cos']) * cos_theta.item()
                state['s'] = s

                exp_avg.mul_(beta1).add_(grad, alpha=1 - beta1)
                exp_avg_sq.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)

                bias_correction1 = 1 - beta1 ** state['step']
                bias_correction2 = 1 - beta2 ** state['step']
                denom = (exp_avg_sq.sqrt() / math.sqrt(bias_correction2)).add_(group['eps'])
                step_size = group['lr'] / bias_correction1

                update = -step_size * exp_avg / denom
                update.mul_(1 + group['c'] * s)

                if group['weight_decay'] != 0:
                    p.add_(p, alpha=-group['weight_decay'] * group['lr'])

                p.add_(update)
                state['prev_grad'].copy_(grad)
        return loss

class Lion(Optimizer):
    def __init__(self, params, lr=1e-4, betas=(0.9, 0.99), weight_decay=0.01):
        defaults = dict(lr=lr, betas=betas, weight_decay=weight_decay)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None: continue
                grad = p.grad
                state = self.state[p]
                if group["weight_decay"] != 0:
                    p.mul_(1 - group["lr"] * group["weight_decay"])
                if "exp_avg" not in state: state["exp_avg"] = torch.zeros_like(p)
                exp_avg = state["exp_avg"]
                beta1, beta2 = group["betas"]
                
                update = exp_avg.mul(beta1).add(grad, alpha=1 - beta1)
                p.add_(torch.sign(update), alpha=-group["lr"])
                exp_avg.mul_(beta2).add_(grad, alpha=1 - beta2)

class SAM(Optimizer):
    def __init__(self, params, base_optimizer, rho=0.01, **kwargs):
        defaults = dict(rho=rho, **kwargs)
        super(SAM, self).__init__(params, defaults)
        self.base_optimizer = base_optimizer(self.param_groups, **kwargs)
        self.param_groups = self.base_optimizer.param_groups

    @torch.no_grad()
    def first_step(self, zero_grad=False):
        grad_norm = self._grad_norm()
        for group in self.param_groups:
            scale = group["rho"] / (grad_norm + 1e-12)
            for p in group["params"]:
                if p.grad is None: continue
                e_w = p.grad * scale.to(p)
                p.add_(e_w)
                self.state[p]["e_w"] = e_w
        if zero_grad: self.zero_grad()

    @torch.no_grad()
    def second_step(self, zero_grad=False):
        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None: continue
                p.sub_(self.state[p]["e_w"])
        self.base_optimizer.step()
        if zero_grad: self.zero_grad()

    def _grad_norm(self):
        norm = torch.norm(torch.stack([
            p.grad.norm(p=2) for group in self.param_groups for p in group["params"] if p.grad is not None
        ]), p=2)
        return norm

# ============================================================================
# 📉 FLAT MINIMA EVALUATION
# ============================================================================
def evaluate_loss_sharpness(model, data_loader, epsilon=0.01):
    # Forced FP32 for Sharpness to avoid backward underflow on unscaled loss
    model.eval()
    original_state = {k: v.clone() for k, v in model.state_dict().items()}
    
    batch = next(iter(data_loader))
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    labels = batch['labels'].to(device)
    
    model.zero_grad()
    
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    loss_orig = F.cross_entropy(outputs.logits, labels)
    if loss_orig.dim() > 0: loss_orig = loss_orig.mean()
        
    loss_orig.backward()
    
    with torch.no_grad():
        for p in model.parameters():
            if p.grad is not None:
                p.add_(epsilon * torch.sign(p.grad))
                
    with torch.no_grad():
        outputs_perturbed = model(input_ids=input_ids, attention_mask=attention_mask)
        loss_perturbed = F.cross_entropy(outputs_perturbed.logits, labels)
        if loss_perturbed.dim() > 0: loss_perturbed = loss_perturbed.mean()
    
    model.load_state_dict(original_state)
    return max(0.0, loss_perturbed.item() - loss_orig.item()) * 100

# ============================================================================
# 📂 DATA LOADER 
# ============================================================================
def load_agnews_data(tokenizer, max_length=128, subset_ratio=1.0, seed=42):
    train_df = pd.read_csv(os.path.join(AG_NEWS_PATH, 'train.csv'))
    test_df = pd.read_csv(os.path.join(AG_NEWS_PATH, 'test.csv'))
    
    for df in [train_df, test_df]:
        if 'Class Index' in df.columns: df.rename(columns={'Class Index': 'label'}, inplace=True)
        if 'Description' in df.columns and 'text' not in df.columns: df.rename(columns={'Description': 'text'}, inplace=True)
        elif 'description' in df.columns and 'text' not in df.columns: df.rename(columns={'description': 'text'}, inplace=True)
        if df['label'].min() == 1: df['label'] -= 1

    class AGNewsDataset(Dataset):
        def __init__(self, dataframe, tokenizer, max_len):
            self.encodings = tokenizer(dataframe['text'].tolist(), truncation=True,
                                       padding='max_length', max_length=max_len, return_tensors='pt')
            self.labels = torch.tensor(dataframe['label'].values)
        def __len__(self): return len(self.labels)
        def __getitem__(self, idx):
            return {'input_ids': self.encodings['input_ids'][idx],
                    'attention_mask': self.encodings['attention_mask'][idx],
                    'labels': self.labels[idx]}

    full_train_ds = AGNewsDataset(train_df, tokenizer, max_length)
    val_ds = AGNewsDataset(test_df, tokenizer, max_length)

    if subset_ratio < 1.0:
        indices = np.random.default_rng(seed).choice(len(full_train_ds), int(len(full_train_ds) * subset_ratio), replace=False)
        train_ds = Subset(full_train_ds, indices)
    else:
        train_ds = full_train_ds

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
    return train_loader, val_loader

# ============================================================================
# 🔄 TRAINING & EVALUATION 
# ============================================================================
def train_and_eval(model, train_loader, val_loader, optimizer, scheduler, num_epochs, is_sam=False):
    criterion = nn.CrossEntropyLoss()
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'epoch_time': [], 'graceful_exit': False}
    base_opt = optimizer.base_optimizer if is_sam else optimizer
    scaler = GradScaler() 

    for epoch in range(num_epochs):
        t0 = time.time()
        model.train()
        tr_loss, tr_correct, total = 0.0, 0, 0
        
        for batch in train_loader:
            input_ids = batch['input_ids'].to(device, non_blocking=True)
            attention_mask = batch['attention_mask'].to(device, non_blocking=True)
            labels = batch['labels'].to(device, non_blocking=True)
            
            if is_sam:
                # 🛡️ SAM BULLETPROOF FIX: Force FP32, Bypass GradScaler completely
                base_opt.zero_grad()
                
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                loss = criterion(outputs.logits, labels)
                if loss.dim() > 0: loss = loss.mean()
                
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_VALUE)
                optimizer.first_step(zero_grad=True)
                
                outputs2 = model(input_ids=input_ids, attention_mask=attention_mask)
                loss2 = criterion(outputs2.logits, labels)
                if loss2.dim() > 0: loss2 = loss2.mean()
                
                loss2.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_VALUE)
                optimizer.second_step()
                
            else:
                # 🚀 STANDARD OPTIMIZERS: High-Speed FP16 with AMP
                optimizer.zero_grad()
                with autocast():
                    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                    loss = criterion(outputs.logits, labels)
                    if loss.dim() > 0: loss = loss.mean()
                
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_VALUE)
                scaler.step(optimizer)
                scaler.update()
            
            scheduler.step()
            preds = outputs.logits.argmax(dim=1)
            tr_loss += loss.item() * labels.size(0)
            tr_correct += preds.eq(labels).sum().item()
            total += labels.size(0)

        epoch_time = time.time() - t0
        history['epoch_time'].append(epoch_time)
        history['train_loss'].append(tr_loss / total)
        history['train_acc'].append(tr_correct / total)

        # Validation (Always FP16 for speed)
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad(), autocast():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device, non_blocking=True)
                attention_mask = batch['attention_mask'].to(device, non_blocking=True)
                labels = batch['labels'].to(device, non_blocking=True)
                
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                loss = criterion(outputs.logits, labels)
                if loss.dim() > 0: loss = loss.mean()
                
                val_loss += loss.item() * labels.size(0)
                val_correct += outputs.logits.argmax(1).eq(labels).sum().item()
                val_total += labels.size(0)
                
        history['val_loss'].append(val_loss / val_total)
        history['val_acc'].append(val_correct / val_total)

        if time.time() - GLOBAL_START_TIME > KAGGLE_TIME_LIMIT:
            print("\n⚠️ KAGGLE TIME LIMIT REACHED! Triggering Graceful Exit...", flush=True)
            history['graceful_exit'] = True
            break

    sharpness = evaluate_loss_sharpness(model, train_loader)

    return {
        'test_acc': history['val_acc'][-1] if history['val_acc'] else 0.0, 
        'sharpness': sharpness, 
        'avg_epoch_time': float(np.mean(history['epoch_time'])) if history['epoch_time'] else 0.0,
        'history': history,
        'graceful_exit': history['graceful_exit']
    }

# ============================================================================
# 🔍 CHECKPOINTING & HP SEARCH 
# ============================================================================
def load_json(filepath):
    if os.path.exists(filepath):
        try:
            with open(filepath, 'r') as f: return json.load(f)
        except: pass
    return {}

def save_json(data, filepath):
    with open(filepath, 'w') as f: json.dump(data, f, indent=4)

def quick_hyperparameter_search(tokenizer, optimizer_name, opt_class, is_sam=False):
    hp_data = load_json(HP_CHECKPOINT_FILE)
    if optimizer_name in hp_data:
        return hp_data[optimizer_name]['lr'], hp_data[optimizer_name]['wd']
        
    train_loader, val_loader = load_agnews_data(tokenizer, MAX_LENGTH, subset_ratio=HP_SUBSET_RATIO, seed=HP_SEARCH_SEED)
    total_steps = HP_EPOCHS * len(train_loader)
    best_acc, best_params = 0.0, (HP_LR_GRID[0], HP_WD_GRID[0])

    for lr in HP_LR_GRID:
        for wd in HP_WD_GRID:
            set_seed(HP_SEARCH_SEED)
            model = BertForSequenceClassification.from_pretrained(BERT_PATH, num_labels=4, local_files_only=True)
            
            if torch.cuda.device_count() > 1:
                model = nn.DataParallel(model)
            model = model.to(device)
            
            if is_sam:
                optimizer = opt_class(model.parameters(), optim.AdamW, lr=lr, weight_decay=wd, rho=0.01)
            else:
                optimizer = opt_class(model.parameters(), lr=lr, weight_decay=wd)
                
            base_opt = optimizer.base_optimizer if is_sam else optimizer
            scheduler = get_cosine_schedule_with_warmup(base_opt, int(WARMUP_RATIO*total_steps), total_steps)
            res = train_and_eval(model, train_loader, val_loader, optimizer, scheduler, HP_EPOCHS, is_sam)
            
            if res['test_acc'] > best_acc:
                best_acc = res['test_acc']
                best_params = (lr, wd)
            
            del model, optimizer; gc.collect(); torch.cuda.empty_cache()
    
    hp_data[optimizer_name] = {'lr': best_params[0], 'wd': best_params[1], 'acc': best_acc}
    save_json(hp_data, HP_CHECKPOINT_FILE)
    return best_params

# ============================================================================
# 📊 STATISTICAL ANALYSIS 
# ============================================================================
def adaptive_ci(data, cl=0.95):
    n = len(data)
    if n < 5:
        mean, std = np.mean(data), np.std(data, ddof=1) if n > 1 else 0.0
        margin = t_dist.ppf((1 + cl) / 2, max(n - 1, 1)) * std / np.sqrt(n) if n > 0 else 0
        return float(mean - margin), float(mean + margin)
    rng = np.random.default_rng(42)
    boot = [np.mean(rng.choice(data, n, replace=True)) for _ in range(1000)]
    alpha = (1 - cl) / 2
    return float(np.percentile(boot, 100 * alpha)), float(np.percentile(boot, 100 * (1 - alpha)))

def compute_statistics(multi_results):
    metrics = ['test_acc', 'sharpness', 'avg_epoch_time']
    agg, var, cvs, ci = {}, {}, {}, {}
    for opt, runs in multi_results.items():
        if not runs: continue
        agg[opt], var[opt], cvs[opt], ci[opt] = {}, {}, {}, {}
        for m in metrics:
            vals = [r[m] for r in runs if m in r]
            if not vals: continue
            mean_v = float(np.mean(vals))
            std_v = float(np.std(vals, ddof=1)) if len(vals) > 1 else 0.0
            agg[opt][f'{m}_mean'], agg[opt][f'{m}_std'] = mean_v, std_v
            var[opt][f'{m}_var'] = float(np.var(vals, ddof=1)) if len(vals) > 1 else 0.0
            ci[opt][m] = adaptive_ci(vals)
    return agg, var, cvs, ci

def statistical_tests(results, baseline='CosAdam'):
    if baseline not in results or len(results[baseline]) < 2: return {}
    base_vals = np.array([r['test_acc'] for r in results[baseline]])
    sig, pvals, basenames = {}, [], []
    
    for opt, runs in results.items():
        if opt == baseline or not runs or len(runs) < 2: continue
        vals = np.array([r['test_acc'] for r in runs])
        if len(base_vals) != len(vals): continue
            
        basenames.append(opt)
        diffs = base_vals - vals
        mean_diff, std_diff = float(np.mean(diffs)), float(np.std(diffs, ddof=1))
        
        if std_diff < 1e-8:
            d_z, p_raw = 0.0, (1.0 if abs(mean_diff) < 1e-8 else 0.0)
        else:
            d_z = float(mean_diff / std_diff)
            try: _, p_raw = ttest_rel(base_vals, vals)
            except: p_raw = 1.0
                
        pvals.append(float(p_raw))
        sig[opt] = {'diff': mean_diff, 'p_raw': float(p_raw), 'es_z': d_z}
        
    if pvals:
        _, p_corr, _, _ = multipletests(pvals, method='fdr_bh')
        for i, opt in enumerate(basenames):
            sig[opt]['p_corr'] = float(p_corr[i])
            sig[opt]['sig'] = '***' if p_corr[i] < 0.001 else '**' if p_corr[i] < 0.01 else '*' if p_corr[i] < 0.05 else 'ns'
    return sig

# ============================================================================
# 📄 VISUALIZATION & LATEX FUNCTIONS
# ============================================================================
def performance_table(agg, ci, caption, label, filename):
    mc = [{'d':'Acc','k':'test_acc','f':'.4f'}, {'d':'Sharpness','k':'sharpness','f':'.4f'},
          {'d':'Time(s)','k':'avg_epoch_time','f':'.2f'}]
    lines = [
        r"\begin{table}[htbp]", r"\centering", r"\caption{" + caption + "}",
        r"\label{" + label + "}", r"\resizebox{\textwidth}{!}{%",
        r"\begin{tabular}{l" + "c" * len(mc) + "}", r"\toprule",
        r"{\bf Optimizer} & " + " & ".join([f"{{\\bf {m['d']}}}" for m in mc]) + r" \\", r"\midrule"
    ]
    for opt in agg.keys():
        row = [opt]
        for m in mc:
            if f"{m['k']}_mean" not in agg[opt]:
                row.append("-")
                continue
            mean, std, (low, high) = agg[opt][f"{m['k']}_mean"], agg[opt][f"{m['k']}_std"], ci[opt][m['k']]
            row.append(f"{mean:{m['f']}} $\\pm$ {std:{m['f']}} \\; [{low:{m['f']}}, {high:{m['f']}}]")
        lines.append(" & ".join(row) + r" \\")
    lines.extend([r"\bottomrule", r"\end{tabular}", r"}", r"\end{table}"])
    with open(os.path.join(RESULTS_DIR, "latex_tables", filename), "w") as f: f.write("\n".join(lines))

def stats_table(sig, caption, label, filename):
    lines = [
        r"\begin{table}[htbp]", r"\centering", r"\caption{" + caption + "}",
        r"\label{" + label + "}", r"\begin{tabular}{lccccc}", r"\toprule",
        r"\textbf{Comparison vs CosAdam} & \textbf{$\Delta$ Acc} & \textbf{$p_{\text{ttest}}$} & \textbf{$p_{\text{FDR}}$} & \textbf{$d_z$} & \textbf{Sig.} \\",
        r"\midrule"
    ]
    for opt, r in sig.items():
        diff_str = f"${r['diff']:+.4f}$" if abs(r['diff']) >= 1e-4 else f"${r['diff']:+.1e}$"
        lines.append(f"{opt} & {diff_str} & {r['p_raw']:.3f} & {r['p_corr']:.3f} & {r['es_z']:.3f} & {r['sig']} \\\\")
    lines.extend([r"\bottomrule", r"\end{tabular}", r"\end{table}"])
    with open(os.path.join(RESULTS_DIR, "latex_tables", filename), "w") as f: f.write("\n".join(lines))

def plot_learning_curves(results, dataset_name="BERT-base on AG News", save_path=None):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
    colors = plt.cm.tab10.colors
    line_styles = ['-', '--']

    for idx, (name, res) in enumerate(results.items()):
        if len(res) == 0: continue

        all_train_acc, all_val_acc = [], []
        all_train_loss, all_val_loss = [], []
        
        valid_accs = [r['history'].get('train_acc', []) for r in res if 'train_acc' in r['history']]
        if not valid_accs: continue
        max_epochs = max([len(a) for a in valid_accs])

        for r in res:
            history = r['history']
            train_acc, val_acc = list(history.get('train_acc', [])), list(history.get('val_acc', []))
            train_loss, val_loss = list(history.get('train_loss', [])), list(history.get('val_loss', []))

            if len(train_acc) < max_epochs and len(train_acc) > 0:
                train_acc += [train_acc[-1]] * (max_epochs - len(train_acc))
                val_acc += [val_acc[-1]] * (max_epochs - len(val_acc))
                train_loss += [train_loss[-1]] * (max_epochs - len(train_loss))
                val_loss += [val_loss[-1]] * (max_epochs - len(val_loss))

            if train_acc: all_train_acc.append(train_acc)
            if val_acc: all_val_acc.append(val_acc)
            if train_loss: all_train_loss.append(train_loss)
            if val_loss: all_val_loss.append(val_loss)

        epochs = range(1, max_epochs + 1)
        clean_name = name.replace('_', ' ')

        if all_train_acc and all_val_acc:
            mean_train_acc, std_train_acc = np.mean(all_train_acc, axis=0), np.std(all_train_acc, axis=0)
            mean_val_acc, std_val_acc = np.mean(all_val_acc, axis=0), np.std(all_val_acc, axis=0)
            
            ax1.plot(epochs, mean_train_acc, color=colors[idx % len(colors)], linewidth=2, linestyle=line_styles[0], alpha=0.5)
            ax1.plot(epochs, mean_val_acc, label=f'{clean_name}', color=colors[idx % len(colors)], linewidth=2, linestyle=line_styles[1])
            ax1.fill_between(epochs, mean_train_acc - std_train_acc, mean_train_acc + std_train_acc, color=colors[idx % len(colors)], alpha=0.1)
            ax1.fill_between(epochs, mean_val_acc - std_val_acc, mean_val_acc + std_val_acc, color=colors[idx % len(colors)], alpha=0.2)

        if all_train_loss and all_val_loss:
            mean_train_loss, std_train_loss = np.mean(all_train_loss, axis=0), np.std(all_train_loss, axis=0)
            mean_val_loss, std_val_loss = np.mean(all_val_loss, axis=0), np.std(all_val_loss, axis=0)

            ax2.plot(epochs, mean_train_loss, color=colors[idx % len(colors)], linewidth=2, linestyle=line_styles[0], alpha=0.5)
            ax2.plot(epochs, mean_val_loss, label=f'{clean_name}', color=colors[idx % len(colors)], linewidth=2, linestyle=line_styles[1])
            ax2.fill_between(epochs, mean_train_loss - std_train_loss, mean_train_loss + std_train_loss, color=colors[idx % len(colors)], alpha=0.1)
            ax2.fill_between(epochs, mean_val_loss - std_val_loss, mean_val_loss + std_val_loss, color=colors[idx % len(colors)], alpha=0.2)

    ax1.set_title(f'{dataset_name} - Accuracy', fontweight='bold')
    ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy'); ax1.grid(True, alpha=0.3)
    ax1.legend(loc='lower right', fontsize=9)

    ax2.set_title(f'{dataset_name} - Loss', fontweight='bold')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss'); ax2.grid(True, alpha=0.3)
    ax2.legend(loc='upper right', fontsize=9)

    num_runs = len(list(results.values())[0]) if results else 0
    guidance_text = (
        f"Shaded areas represent ±1 standard deviation across {num_runs} independent runs.\n"
        "Solid lines: Training, Dashed lines: Validation"
    )
    plt.figtext(0.5, 0.01, guidance_text, ha='center', fontsize=11, bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))
    plt.tight_layout(rect=[0, 0.05, 1, 1])
    
    if not save_path: save_path = os.path.join(RESULTS_DIR, "plots", "nlp_main_learning_curves.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

# ============================================================================
# 📦 FINAL ZIP EXPORT
# ============================================================================
def create_final_zip():
    zip_path = "/kaggle/working/cosadam_nlp_results.zip"
    base_folder = os.path.basename(RESULTS_DIR)
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        if os.path.exists(MAIN_CHECKPOINT_FILE):
            zipf.write(MAIN_CHECKPOINT_FILE, arcname=os.path.join(base_folder, os.path.basename(MAIN_CHECKPOINT_FILE)))
        for sub_dir in ["plots", "latex_tables"]:
            dir_path = os.path.join(RESULTS_DIR, sub_dir)
            if os.path.isdir(dir_path):
                for root, _, files in os.walk(dir_path):
                    for file in files:
                        zipf.write(os.path.join(root, file), arcname=os.path.join(base_folder, sub_dir, file))
    print(f"\n📦 Final results safely archived at: {zip_path}", flush=True)

# ============================================================================
# 🚀 MAIN EXECUTION
# ============================================================================
if __name__ == "__main__":
    print("CosAdam NLP COMPREHENSIVE COMPARISON (BERT-base on AG News)")
    tokenizer = BertTokenizerFast.from_pretrained(BERT_PATH, local_files_only=True)

    hp_configs = {
        'CosAdam': (CosAdam, *quick_hyperparameter_search(tokenizer, 'CosAdam', CosAdam), False),
        'AdamW': (optim.AdamW, *quick_hyperparameter_search(tokenizer, 'AdamW', optim.AdamW), False),
        'Lion': (Lion, *quick_hyperparameter_search(tokenizer, 'Lion', Lion), False),
        'SAM': (SAM, *quick_hyperparameter_search(tokenizer, 'SAM', SAM, is_sam=True), True)
    }

    train_loader, val_loader = load_agnews_data(tokenizer, MAX_LENGTH, subset_ratio=1.0)
    total_steps = NLP_EPOCHS * len(train_loader)
    results = load_json(MAIN_CHECKPOINT_FILE)
    for name in hp_configs:
        if name not in results: results[name] = []

    graceful_exit_triggered = False

    for seed in MAIN_SEEDS:
        if graceful_exit_triggered: break
        
        for opt_name, (opt_class, lr, wd, is_sam) in hp_configs.items():
            if len([r for r in results[opt_name] if r.get('seed') == seed]) > 0: continue
                
            set_seed(seed)
            model = BertForSequenceClassification.from_pretrained(BERT_PATH, num_labels=4, local_files_only=True)
            
            if torch.cuda.device_count() > 1:
                model = nn.DataParallel(model)
            model = model.to(device)
            
            if is_sam:
                optimizer = opt_class(model.parameters(), optim.AdamW, lr=lr, weight_decay=wd, rho=0.01)
            else:
                optimizer = opt_class(model.parameters(), lr=lr, weight_decay=wd)
                
            base_opt = optimizer.base_optimizer if is_sam else optimizer
            scheduler = get_cosine_schedule_with_warmup(base_opt, int(WARMUP_RATIO*total_steps), total_steps)
            
            print(f"🔥 Evaluating {opt_name} on seed={seed}...", flush=True)
            res = train_and_eval(model, train_loader, val_loader, optimizer, scheduler, NLP_EPOCHS, is_sam)
            res['seed'] = seed
            print(f"[{opt_name} | Seed {seed}] Acc: {res['test_acc']:.4f} | Sharpness: {res['sharpness']:.4f}")
            
            results[opt_name].append(res)
            save_json(results, MAIN_CHECKPOINT_FILE)
            
            del model, optimizer; gc.collect(); torch.cuda.empty_cache()
            
            if res.get('graceful_exit', False):
                graceful_exit_triggered = True
                break

    print("\nCompiling statistical backends, LaTeX tables, and high-res visualizations...", flush=True)
    agg, var, cvs, ci = compute_statistics(results)
    
    if agg:
        performance_table(agg, ci, r"NLP Main Comparison (BERT-base on AG News, Mean $\pm$ SD [95\% CI])", 
                          "tab:nlp_main_perf", "t1_nlp_main_perf.tex")
        sig = statistical_tests(results, baseline='CosAdam')
        stats_table(sig, r"Statistical Significance (CosAdam vs baselines, Paired T-Test) - BERT-base on AG News", 
                    "tab:nlp_main_stats", "t2_nlp_main_stats.tex")
        
        plot_learning_curves(results, dataset_name="BERT-base on AG News")
    
    create_final_zip()
    print("\n✅ NLP MAIN COMPARISON COMPLETE.")

CosAdam NLP COMPREHENSIVE COMPARISON (BERT-base on AG News)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: /kaggle/input/datasets/virajjayant/bertbaseuncased/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: /kaggle/input/datasets/virajjayant/bertbaseuncased/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because

...

BertForSequenceClassification LOAD REPORT from: /kaggle/input/datasets/virajjayant/bertbaseuncased/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because

🔥 Evaluating Lion on seed=587...
[Lion | Seed 587] Acc: 0.8846 | Sharpness: 353.3546


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: /kaggle/input/datasets/virajjayant/bertbaseuncased/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because

🔥 Evaluating SAM on seed=587...

⚠️ KAGGLE TIME LIMIT REACHED! Triggering Graceful Exit...
[SAM | Seed 587] Acc: 0.9382 | Sharpness: 257.9564

Compiling statistical backends, LaTeX tables, and high-res visualizations...

📦 Final results safely archived at: /kaggle/working/cosadam_nlp_results.zip

✅ NLP MAIN COMPARISON COMPLETE.
